# 🌸 SaralGati: Fine-Tune Llama-3.1-8B on Kaggle Dual T4 (2x T4 GPUs)

This notebook trains a custom **LoRA Adapter** on **Meta-Llama-3.1-8B-Instruct** using Kaggle's **Dual T4 GPUs (2x T4)** with PyTorch Distributed Data Parallel (`torchrun`).

### ⚡ Hardware Setting on Kaggle:
- In the right sidebar: **Accelerator -> GPU T4 x 2**
- Internet: **Internet On** (Required to download packages and model weights)
- Training speed: **~2x faster** using both GPUs simultaneously!

## 1. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

## 2. Generate Multi-GPU Training Script (`train.py`)
This writes a dedicated standalone script so `torchrun` can run distributed training across both T4 GPUs.

In [ ]:
%%writefile train.py
import os
import json
import urllib.request
import torch
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import FastLanguageModel, is_bfloat16_supported

# 1. Download dataset if not present
dataset_file = "saralgati_popular_apps_train.jsonl"
if not os.path.exists(dataset_file):
    url = "https://raw.githubusercontent.com/ShunyaPulse/SaralGati/main/saralgati_popular_apps_train.jsonl"
    print(f"Downloading dataset from GitHub...")
    urllib.request.urlretrieve(url, dataset_file)

data = []
with open(dataset_file, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

print(f"Loaded {len(data)} training samples.")

# 2. Load Model & Setup LoRA
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# Format prompts with Llama 3.1 chat template
def format_prompts(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompts, batched = True)

# 3. Configure Trainer for Dual T4 (DDP)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,  # 2 per GPU = 4 total effective batch
        gradient_accumulation_steps = 2,
        warmup_steps = 15,
        max_steps = 250,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting Multi-GPU Training...")
trainer.train()

# 4. Save clean LoRA adapter (rank 0 only)
local_rank = int(os.environ.get("LOCAL_RANK", 0))
if local_rank == 0:
    output_dir = "saralgati_llama31_8b_lora"
    os.makedirs(output_dir, exist_ok = True)
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    # Patch adapter_config.json for Cloudflare Workers AI
    config_path = os.path.join(output_dir, "adapter_config.json")
    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)
    config["base_model_name_or_path"] = "meta-llama/Llama-3.1-8B-Instruct"
    config["model_type"] = "llama"
    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)
    print("Adapter successfully saved and patched for Cloudflare Workers AI!")


## 3. Launch Multi-GPU Training (`torchrun` on 2x T4 GPUs)
This runs both T4 GPUs at full capacity simultaneously.

In [ ]:
!torchrun --nproc_per_node=2 train.py

## 4. Zip LoRA Adapter (~100MB)

In [ ]:
import shutil, os
output_dir = "saralgati_llama31_8b_lora"
zip_filename = "saralgati_llama31_8b_lora"
shutil.make_archive(zip_filename, "zip", output_dir)
print(f"✅ Archive created: {zip_filename}.zip ({os.path.getsize(zip_filename + '.zip') / (1024*1024):.2f} MB)")

## 5. 🚀 1-Click Direct Upload to Cloudflare Workers AI

In [ ]:
import requests, getpass

ACCOUNT_ID = input("Enter Cloudflare Account ID (default: 7953d66fe9e6158b01faf0752ae8c841): ").strip() or "7953d66fe9e6158b01faf0752ae8c841"
API_TOKEN = getpass.getpass("Paste Cloudflare API Token: ").strip()
FINE_TUNE_NAME = "saralgati-elder-llama31-8b"
DESCRIPTION = "SaralGati Elder-Friendly Hindi Companion on Llama-3.1-8B-Instruct"
output_dir = "saralgati_llama31_8b_lora"

headers = {"Authorization": f"Bearer {API_TOKEN}"}

print(f"Step 1: Finding fine-tune '{FINE_TUNE_NAME}' on Cloudflare...")
list_url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/finetunes"
list_res = requests.get(list_url, headers=headers)

finetune_id = None
if list_res.ok:
    for ft in list_res.json().get("result", []):
        if ft.get("name") == FINE_TUNE_NAME:
            finetune_id = ft.get("id")
            print(f"Found existing fine-tune ID: {finetune_id}")
            break

if not finetune_id:
    create_payload = {
        "name": FINE_TUNE_NAME,
        "description": DESCRIPTION,
        "model": "@cf/meta/llama-guard-3-8b"
    }
    res = requests.post(list_url, headers=headers, json=create_payload)
    if res.ok:
        finetune_id = res.json().get("result", {}).get("id")
        print(f"Created new fine-tune ID: {finetune_id}")

# If fine-tune already had assets, clean old fine-tune and recreate to avoid MAX_ASSETS_ERROR
if finetune_id:
    print(f"Refreshing fine-tune to avoid MAX_ASSETS_ERROR...")
    del_url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/finetunes/{finetune_id}"
    requests.delete(del_url, headers=headers)
    
    create_payload = {
        "name": FINE_TUNE_NAME,
        "description": DESCRIPTION,
        "model": "@cf/meta/llama-guard-3-8b"
    }
    res = requests.post(list_url, headers=headers, json=create_payload)
    finetune_id = res.json().get("result", {}).get("id")
    print(f"Fresh fine-tune ID ready: {finetune_id}")

if finetune_id:
    print(f"\nStep 2: Uploading adapter assets to fine-tune '{FINE_TUNE_NAME}'...")
    upload_url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/finetunes/{finetune_id}/finetune-assets"

    # 1. Upload adapter_config.json
    with open(f"{output_dir}/adapter_config.json", "rb") as f:
        res1 = requests.post(upload_url, headers=headers, data={"file_name": "adapter_config.json"}, files={"file": ("adapter_config.json", f, "application/json")})
        print("adapter_config.json upload:", res1.status_code)

    # 2. Upload adapter_model.safetensors
    with open(f"{output_dir}/adapter_model.safetensors", "rb") as f:
        res2 = requests.post(upload_url, headers=headers, data={"file_name": "adapter_model.safetensors"}, files={"file": ("adapter_model.safetensors", f, "application/octet-stream")})
        print("adapter_model.safetensors upload:", res2.status_code)

    if res1.ok and res2.ok:
        print("\n🎉🎉 SUCCESS! LoRA adapter is fully deployed and ready on Cloudflare Workers AI!")
    else:
        print("\n⚠️ Upload finished with warnings. Check logs above.")
else:
    print("❌ Could not obtain fine-tune ID.")